# Hyperparameter Tuning with Keras Tuner
## TensorVerseHub — Supplementary Series

---

### What You'll Learn

| Topic | Description |
|---|---|
| **Why Tune?** | Impact of hyperparameters on model quality |
| **Keras Tuner API** | `HyperModel`, `hp` object, search spaces |
| **RandomSearch** | Baseline tuner — broad exploration |
| **Hyperband** | Efficient early-stopping-based search |
| **BayesianOptimization** | Probabilistic surrogate model search |
| **Custom HyperModel** | OOP interface for reusable architectures |
| **Tuning CNNs** | Architecture + training hyperparameters |
| **Tuning Transformers** | Embedding dim, heads, ff layers |
| **Learning Rate Schedules** | Tuning schedule type and shape |
| **Results Analysis** | Comparing trials, visualising loss surfaces |
| **Best Practices** | Avoiding over-tuning, search budget planning |

### Prerequisites
- Notebooks 04 — Keras Sequential / Functional APIs
- Notebook 06 — Keras Callbacks & Optimisation
- Basic understanding of model training loops

```
pip install keras-tuner
```

In [ ]:
# ── Environment Setup ────────────────────────────────────────────────────────
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    import keras_tuner as kt
except ImportError:
    _install("keras-tuner")
    import keras_tuner as kt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json, os, warnings
warnings.filterwarnings("ignore")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print(f"TensorFlow : {tf.__version__}")
print(f"Keras Tuner: {kt.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")

---

## Section 1: Why Hyperparameter Tuning Matters

A model's **hyperparameters** are configuration choices made *before* training begins — they cannot be learned from data by gradient descent. Poor choices can make an otherwise excellent architecture completely fail.

### Common Hyperparameters

| Category | Examples |
|---|---|
| **Architecture** | Number of layers, units per layer, activation functions |
| **Regularisation** | Dropout rate, L2 weight decay, batch normalisation |
| **Optimiser** | Type (Adam/SGD/RMSprop), learning rate, momentum |
| **Training** | Batch size, number of epochs, warmup steps |
| **Data** | Augmentation strength, sequence length |

### Manual vs. Automated Tuning

```
Manual search:   Engineer intuition + grid of educated guesses
Grid search:     Exhaustive — exponential cost, inflexible
Random search:   Sample at random — surprisingly competitive (Bergstra & Bengio, 2012)
Hyperband:       Multi-armed bandit — early-stop bad configs early (Li et al., 2017)
Bayesian Opt.:   Build a probabilistic surrogate — exploit past knowledge
```

**Key insight from Bergstra & Bengio 2012:** Only a *few* hyperparameters matter strongly for any given problem. Random search naturally allocates more trials to the important dimensions.

In [ ]:
# ── Dataset: CIFAR-10 (used throughout this notebook) ────────────────────────
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalise to [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test        = x_test.astype("float32") / 255.0

# Small subset for fast tuning demonstrations (use full dataset in production)
TRAIN_SIZE  = 5000
VAL_SIZE    = 1000

x_train = x_train_full[:TRAIN_SIZE]
y_train = y_train_full[:TRAIN_SIZE]
x_val   = x_train_full[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
y_val   = y_train_full[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]

CLASS_NAMES = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]

print(f"Train : {x_train.shape}  |  Val : {x_val.shape}  |  Test : {x_test.shape}")

# Quick preview
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[y_train[i, 0]], fontsize=8)
    ax.axis("off")
plt.suptitle("CIFAR-10 Samples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 2: The Keras Tuner API

### Core Concepts

```
kt.HyperParameters (hp)
    └── hp.Int(name, min, max, step)         # integer range
    └── hp.Float(name, min, max, sampling)   # float range (log/linear/reverse_log)
    └── hp.Choice(name, values)              # discrete set
    └── hp.Boolean(name)                     # True / False
    └── hp.Fixed(name, value)                # constant (for ablations)

Tuner subclasses
    ├── kt.RandomSearch         — random samples from the search space
    ├── kt.Hyperband            — successive halving with early stopping
    ├── kt.BayesianOptimization — Gaussian process surrogate
    └── kt.Sklearn              — wraps scikit-learn estimators

HyperModel
    └── build(hp) → tf.keras.Model    # define architecture using hp
    └── fit(hp, model, *args, **kw)   # optional: tune training logic too
```

### Defining a Search Space — Two styles

**Inline (simple):**
```python
def model_fn(hp):
    units = hp.Int("units", 32, 256, step=32)
    ...
    return model
```

**Class-based (reusable):**
```python
class MyHyperModel(kt.HyperModel):
    def build(self, hp): ...
    def fit(self, hp, model, *args, **kwargs): ...
```

In [ ]:
# ── Exploring the hp object interactively ────────────────────────────────────
hp_demo = kt.HyperParameters()

# Demonstrate every hp type
units      = hp_demo.Int("units",       min_value=32,  max_value=256, step=32)
dropout    = hp_demo.Float("dropout",   min_value=0.0, max_value=0.5, step=0.1)
lr         = hp_demo.Float("lr",        min_value=1e-4, max_value=1e-2, sampling="log")
activation = hp_demo.Choice("activation", ["relu", "gelu", "swish"])
use_bn     = hp_demo.Boolean("use_batch_norm")
fixed_val  = hp_demo.Fixed("num_classes", 10)

print("── Sample values from hp object ──")
print(f"  units         : {units}")
print(f"  dropout       : {dropout}")
print(f"  learning rate : {lr:.6f}")
print(f"  activation    : {activation}")
print(f"  use_batch_norm: {use_bn}")
print(f"  num_classes   : {fixed_val}  (fixed)")

print(f"\n── Registered hp space ({len(hp_demo.space)} entries) ──")
for entry in hp_demo.space:
    print(f"  {entry.name:20s} [{entry.__class__.__name__}]")

---

## Section 3: RandomSearch — Baseline Tuner

**RandomSearch** independently samples hyperparameter configurations at random. Despite its simplicity, it is a strong baseline for low-to-moderate dimensional search spaces.

**When to use:** Quick first pass, identifying which hyperparameters matter most.

### Search space for a Dense baseline on CIFAR-10

We'll tune:
- Number of Dense layers (1–3)
- Units per layer (32–512)
- Activation function
- Dropout rate
- Learning rate (log scale)

In [ ]:
# ── Define the model-building function ───────────────────────────────────────
def build_mlp(hp):
    """MLP for flattened CIFAR-10 — tunes depth, width, activation, dropout, lr."""
    model = keras.Sequential()
    model.add(layers.Flatten(input_shape=(32, 32, 3)))

    # Tune number of Dense layers
    for i in range(hp.Int("num_layers", 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f"units_{i}", min_value=64, max_value=512, step=64),
            activation=hp.Choice("activation", ["relu", "gelu", "swish"]),
        ))
        if hp.Boolean(f"batch_norm_{i}"):
            model.add(layers.BatchNormalization())
        model.add(layers.Dropout(
            rate=hp.Float(f"dropout_{i}", 0.0, 0.5, step=0.1)
        ))

    model.add(layers.Dense(10, activation="softmax"))

    lr = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ── RandomSearch tuner ────────────────────────────────────────────────────────
random_tuner = kt.RandomSearch(
    hypermodel=build_mlp,
    objective="val_accuracy",
    max_trials=12,
    seed=42,
    project_name="mlp_random_search",
    directory="kt_results",
    overwrite=True,
)

random_tuner.search_space_summary()

In [ ]:
# ── Run RandomSearch ─────────────────────────────────────────────────────────
stop_early = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)

random_tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[stop_early],
    verbose=0,
)

print("\n── Top 3 RandomSearch trials ──")
random_tuner.results_summary(num_trials=3)

In [ ]:
# ── Inspect best hyperparameters and retrain ─────────────────────────────────
best_hp_random = random_tuner.get_best_hyperparameters(1)[0]

print("── Best hyperparameters (RandomSearch) ──")
for key, val in best_hp_random.values.items():
    print(f"  {key:30s}: {val}")

# Build and evaluate the best model
best_model_random = random_tuner.get_best_models(1)[0]
loss, acc = best_model_random.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy (RandomSearch best): {acc:.4f}")

---

## Section 4: Hyperband — Efficient Resource Allocation

**Hyperband** (Li et al., 2017) extends the *successive halving* algorithm. It starts many configurations with a small budget, keeps only the best half, doubles the budget, and repeats — finding good configs much faster than random search.

### Key parameters

| Parameter | Meaning |
|---|---|
| `max_epochs` | Maximum training epochs any trial can run |
| `factor` | Bracket reduction factor (default 3) |
| `hyperband_iterations` | How many full Hyperband brackets to run |

### How it works
```
Round 1:  32 configs × 1 epoch   → keep top 1/3 (≈10 configs)
Round 2:  10 configs × 3 epochs  → keep top 1/3 (≈3 configs)
Round 3:   3 configs × 9 epochs  → keep top 1/3 (≈1 config)
Final:     1 config  × 27 epochs → winner
```

This achieves the same quality as training all configs to completion, at a fraction of the cost.

In [ ]:
# ── CNN HyperModel for Hyperband ─────────────────────────────────────────────
class CNNHyperModel(kt.HyperModel):
    """Tunes a small CNN: conv blocks, filters, pooling, dense head, lr."""

    def build(self, hp):
        inputs = keras.Input(shape=(32, 32, 3))
        x = inputs

        # Convolutional blocks
        for i in range(hp.Int("num_conv_blocks", 1, 3)):
            filters = hp.Int(f"filters_{i}", 32, 128, step=32)
            x = layers.Conv2D(filters, 3, padding="same",
                              activation=hp.Choice("conv_activation", ["relu", "swish"]))(x)
            x = layers.BatchNormalization()(x)
            if hp.Boolean(f"extra_conv_{i}"):
                x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
            x = layers.MaxPooling2D()(x)
            x = layers.Dropout(hp.Float(f"conv_dropout_{i}", 0.0, 0.3, step=0.1))(x)

        x = layers.GlobalAveragePooling2D()(x)

        # Dense head
        x = layers.Dense(
            hp.Int("dense_units", 64, 256, step=64),
            activation="relu",
        )(x)
        x = layers.Dropout(hp.Float("dense_dropout", 0.1, 0.5, step=0.1))(x)

        outputs = layers.Dense(10, activation="softmax")(x)

        lr = hp.Float("learning_rate", 1e-4, 3e-3, sampling="log")
        model = keras.Model(inputs, outputs)
        model.compile(
            optimizer=keras.optimizers.Adam(lr),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model


# ── Hyperband tuner ───────────────────────────────────────────────────────────
hyperband_tuner = kt.Hyperband(
    hypermodel=CNNHyperModel(),
    objective="val_accuracy",
    max_epochs=15,
    factor=3,
    hyperband_iterations=1,
    seed=42,
    project_name="cnn_hyperband",
    directory="kt_results",
    overwrite=True,
)

hyperband_tuner.search_space_summary()

In [ ]:
# ── Run Hyperband search ──────────────────────────────────────────────────────
hyperband_tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    callbacks=[
        keras.callbacks.EarlyStopping("val_accuracy", patience=3),
        keras.callbacks.TensorBoard("kt_results/tb_hyperband"),
    ],
    verbose=0,
)

print("\n── Top 3 Hyperband trials ──")
hyperband_tuner.results_summary(num_trials=3)

best_hp_hb = hyperband_tuner.get_best_hyperparameters(1)[0]
best_model_hb = hyperband_tuner.get_best_models(1)[0]
_, acc_hb = best_model_hb.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy (Hyperband best): {acc_hb:.4f}")

---

## Section 5: BayesianOptimization — Smarter Search

**Bayesian optimisation** fits a *Gaussian Process* surrogate model to the (hyperparameter → validation metric) landscape observed so far. The **acquisition function** (Expected Improvement) balances:

- **Exploitation:** Try configs predicted to be good
- **Exploration:** Try configs where the GP is uncertain

This is especially powerful when each trial is expensive (e.g., large models or datasets).

### Acquisition function intuition

```
         │          predicted μ ± 2σ
  metric │   ●                  ░░░░
         │      ●    ●   ░░░░░░░░░░░░░
         │               ░░░░░░░░░░░░░░░░░
         └─────────────────────────────────── hp value
                        ↑ explore here (high uncertainty)
```

Keras Tuner uses a GP over the observed trials — no manual acquisition tuning needed.

In [ ]:
# ── Bayesian Optimization tuner ───────────────────────────────────────────────
bayes_tuner = kt.BayesianOptimization(
    hypermodel=CNNHyperModel(),
    objective="val_accuracy",
    max_trials=15,
    num_initial_points=5,   # random exploration before GP kicks in
    seed=42,
    project_name="cnn_bayesian",
    directory="kt_results",
    overwrite=True,
)

bayes_tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping("val_accuracy", patience=3)],
    verbose=0,
)

print("── Top 3 BayesianOptimization trials ──")
bayes_tuner.results_summary(num_trials=3)

best_model_bayes = bayes_tuner.get_best_models(1)[0]
_, acc_bayes = best_model_bayes.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy (Bayesian best): {acc_bayes:.4f}")

---

## Section 6: Tuning the Learning Rate Schedule

The learning rate schedule is often the single most impactful hyperparameter. Keras Tuner allows you to make even the **schedule type** a searchable hyperparameter by overriding `HyperModel.fit()`.

### Schedules we'll compare

| Schedule | Behaviour |
|---|---|
| `constant` | Fixed LR throughout |
| `cosine_decay` | Smooth cosine annealing |
| `exponential_decay` | Multiply by factor every N steps |
| `warmup_cosine` | Linear warmup + cosine decay |

In [ ]:
# ── HyperModel with tunable learning rate schedule ───────────────────────────
class LRScheduleHyperModel(kt.HyperModel):
    """
    Overrides .fit() to make the LR schedule itself a hyperparameter.
    The model architecture is fixed; only training dynamics are tuned.
    """

    def build(self, hp):
        """Fixed small CNN."""
        inputs = keras.Input(shape=(32, 32, 3))
        x = layers.Conv2D(64, 3, padding="same", activation="relu")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(128, activation="relu")(x)
        x = layers.Dropout(0.3)(x)
        outputs = layers.Dense(10, activation="softmax")(x)
        # Optimiser compiled in fit() so schedule can be set per trial
        return keras.Model(inputs, outputs)

    def fit(self, hp, model, x, y, validation_data, **kwargs):
        base_lr    = hp.Float("base_lr", 1e-4, 1e-2, sampling="log")
        schedule   = hp.Choice("lr_schedule", ["constant", "cosine_decay", "exponential_decay"])
        batch_size = hp.Int("batch_size", 32, 128, step=32)
        epochs     = kwargs.get("epochs", 10)
        steps_per_epoch = len(x) // batch_size

        if schedule == "constant":
            lr = base_lr
        elif schedule == "cosine_decay":
            lr = keras.optimizers.schedules.CosineDecay(
                initial_learning_rate=base_lr,
                decay_steps=steps_per_epoch * epochs,
            )
        else:  # exponential_decay
            lr = keras.optimizers.schedules.ExponentialDecay(
                initial_learning_rate=base_lr,
                decay_steps=steps_per_epoch,
                decay_rate=hp.Float("exp_decay_rate", 0.85, 0.99, step=0.02),
            )

        model.compile(
            optimizer=keras.optimizers.Adam(lr),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )

        return model.fit(
            x, y,
            batch_size=batch_size,
            validation_data=validation_data,
            **{k: v for k, v in kwargs.items() if k != "batch_size"},
        )


# ── Tuner ─────────────────────────────────────────────────────────────────────
lr_tuner = kt.RandomSearch(
    hypermodel=LRScheduleHyperModel(),
    objective="val_accuracy",
    max_trials=10,
    seed=42,
    project_name="lr_schedule_search",
    directory="kt_results",
    overwrite=True,
)

lr_tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    callbacks=[keras.callbacks.EarlyStopping("val_accuracy", patience=3)],
    verbose=0,
)

print("── Best LR schedule hyperparameters ──")
best_lr_hp = lr_tuner.get_best_hyperparameters(1)[0]
for k, v in best_lr_hp.values.items():
    print(f"  {k:25s}: {v}")

---

## Section 7: Analysing Tuning Results

Good tuning practice requires understanding *why* certain configurations win — not just retrieving the best one.

### What to analyse
1. **Trial score distribution** — is there a clear winner or wide variance?
2. **Hyperparameter importance** — which hp moves the needle most?
3. **Learning curves per trial** — are high-scoring trials converging cleanly?
4. **Correlation plots** — how does each hp correlate with validation accuracy?

In [ ]:
# ── Extract all trials from the Hyperband tuner ───────────────────────────────
def extract_trials(tuner):
    """Return list of (hp_dict, val_accuracy) for every completed trial."""
    records = []
    for trial in tuner.oracle.trials.values():
        if trial.score is None:
            continue
        row = dict(trial.hyperparameters.values)
        row["val_accuracy"] = trial.score
        records.append(row)
    return records

records = extract_trials(hyperband_tuner)

# ── 1. Trial score distribution ───────────────────────────────────────────────
scores = [r["val_accuracy"] for r in records]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores, bins=10, color="#E64A19", edgecolor="white", linewidth=0.8)
axes[0].axvline(max(scores), color="gold", linestyle="--", linewidth=2, label=f"Best: {max(scores):.4f}")
axes[0].set_xlabel("Val Accuracy")
axes[0].set_ylabel("# Trials")
axes[0].set_title("Hyperband — Trial Score Distribution")
axes[0].legend()

sorted_scores = sorted(scores, reverse=True)
axes[1].plot(range(1, len(sorted_scores) + 1), sorted_scores, "o-", color="#E64A19", markersize=5)
axes[1].set_xlabel("Trial rank")
axes[1].set_ylabel("Val Accuracy")
axes[1].set_title("Sorted Trial Performance")

plt.tight_layout()
plt.show()
print(f"  Trials run   : {len(scores)}")
print(f"  Best val acc : {max(scores):.4f}")
print(f"  Worst val acc: {min(scores):.4f}")
print(f"  Std dev      : {np.std(scores):.4f}")

In [ ]:
# ── 2. Hyperparameter correlation with val_accuracy ───────────────────────────
import pandas as pd

df = pd.DataFrame(records)

# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "val_accuracy"]

if numeric_cols:
    correlations = df[numeric_cols + ["val_accuracy"]].corr()["val_accuracy"].drop("val_accuracy")
    correlations = correlations.sort_values(key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(10, max(3, len(correlations) * 0.4)))
    colors = ["#E64A19" if v > 0 else "#1565C0" for v in correlations]
    ax.barh(correlations.index, correlations.values, color=colors, edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Pearson Correlation with val_accuracy")
    ax.set_title("Hyperparameter Importance (Correlation Proxy)")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric hyperparameters found in this search — try a tuner with more numeric hps.")

In [ ]:
# ── 3. Tuner comparison: RandomSearch vs Hyperband vs Bayesian ────────────────
def get_best_score(tuner):
    scores = [t.score for t in tuner.oracle.trials.values() if t.score is not None]
    return max(scores) if scores else 0.0

def get_n_trials(tuner):
    return sum(1 for t in tuner.oracle.trials.values() if t.score is not None)

tuners_info = {
    "RandomSearch\n(MLP)":       random_tuner,
    "Hyperband\n(CNN)":          hyperband_tuner,
    "Bayesian\n(CNN)":           bayes_tuner,
    "LR Schedule\n(RandomSearch)": lr_tuner,
}

best_scores = {name: get_best_score(t) for name, t in tuners_info.items()}
n_trials    = {name: get_n_trials(t)   for name, t in tuners_info.items()}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

names = list(best_scores.keys())
vals  = list(best_scores.values())
bars1 = ax1.bar(names, vals, color=sns.color_palette("husl", len(names)), edgecolor="white")
ax1.set_ylabel("Best Val Accuracy")
ax1.set_title("Best Accuracy per Tuner")
ax1.set_ylim(0, 1)
for bar, val in zip(bars1, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.4f}", ha="center", va="bottom", fontsize=9)

bars2 = ax2.bar(names, list(n_trials.values()),
                color=sns.color_palette("husl", len(names)), edgecolor="white")
ax2.set_ylabel("Number of Trials Run")
ax2.set_title("Trials Consumed per Tuner")
for bar, val in zip(bars2, n_trials.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha="center", va="bottom", fontsize=9)

plt.suptitle("Tuner Strategy Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 8: Tuning a Transformer-based Text Classifier

Transformers introduce a richer architectural search space: embedding dimension, number of heads, feed-forward width, number of encoder blocks, and positional encoding style. We tune a compact text classifier on the IMDB sentiment dataset.

In [ ]:
# ── IMDB dataset (prepared for transformer input) ────────────────────────────
VOCAB_SIZE  = 10000
MAX_LEN     = 128
EMBED_DIM   = 64   # used as baseline; tuner will vary this

(x_imdb_train, y_imdb_train), (x_imdb_test, y_imdb_test) = (
    tf.keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
)
x_imdb_train = tf.keras.preprocessing.sequence.pad_sequences(x_imdb_train, maxlen=MAX_LEN)
x_imdb_test  = tf.keras.preprocessing.sequence.pad_sequences(x_imdb_test,  maxlen=MAX_LEN)

# Small subset for fast tuning
x_imdb_tr  = x_imdb_train[:3000]
y_imdb_tr  = y_imdb_train[:3000]
x_imdb_val = x_imdb_train[3000:4000]
y_imdb_val = y_imdb_train[3000:4000]

print(f"IMDB train: {x_imdb_tr.shape}  val: {x_imdb_val.shape}  test: {x_imdb_test.shape}")

In [ ]:
# ── Transformer HyperModel ───────────────────────────────────────────────────
class TransformerHyperModel(kt.HyperModel):
    """
    Tunable Transformer encoder for binary text classification.

    Search space:
      - embed_dim   : 32 / 64 / 128
      - num_heads   : 1 / 2 / 4  (must divide embed_dim)
      - ff_dim      : 64 / 128 / 256 / 512
      - num_blocks  : 1 / 2
      - dropout     : 0.1 – 0.4
      - learning_rate (log scale)
    """

    def build(self, hp):
        embed_dim = hp.Choice("embed_dim", [32, 64, 128])

        # num_heads must divide embed_dim
        valid_heads = [h for h in [1, 2, 4] if embed_dim % h == 0]
        num_heads   = hp.Choice("num_heads", valid_heads)

        ff_dim     = hp.Choice("ff_dim", [64, 128, 256, 512])
        num_blocks = hp.Int("num_blocks", 1, 2)
        dropout    = hp.Float("dropout", 0.1, 0.4, step=0.1)
        lr         = hp.Float("learning_rate", 1e-4, 5e-3, sampling="log")

        # ── Model ──────────────────────────────────────────────────────────────
        inputs = keras.Input(shape=(MAX_LEN,))
        x = layers.Embedding(VOCAB_SIZE, embed_dim)(inputs)

        for _ in range(num_blocks):
            # Multi-head self-attention
            attn_out = layers.MultiHeadAttention(
                num_heads=num_heads, key_dim=embed_dim // num_heads
            )(x, x)
            attn_out = layers.Dropout(dropout)(attn_out)
            x = layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

            # Feed-forward
            ff_out = layers.Dense(ff_dim, activation="relu")(x)
            ff_out = layers.Dense(embed_dim)(ff_out)
            ff_out = layers.Dropout(dropout)(ff_out)
            x = layers.LayerNormalization(epsilon=1e-6)(x + ff_out)

        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(1, activation="sigmoid")(x)

        model = keras.Model(inputs, outputs)
        model.compile(
            optimizer=keras.optimizers.Adam(lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        return model


# ── Hyperband tuner for transformer ──────────────────────────────────────────
transformer_tuner = kt.Hyperband(
    hypermodel=TransformerHyperModel(),
    objective="val_accuracy",
    max_epochs=10,
    factor=3,
    seed=42,
    project_name="transformer_hyperband",
    directory="kt_results",
    overwrite=True,
)

transformer_tuner.search(
    x_imdb_tr, y_imdb_tr,
    validation_data=(x_imdb_val, y_imdb_val),
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping("val_accuracy", patience=2)],
    verbose=0,
)

print("── Top 3 Transformer Hyperband trials ──")
transformer_tuner.results_summary(num_trials=3)

best_transformer = transformer_tuner.get_best_models(1)[0]
_, acc_transformer = best_transformer.evaluate(x_imdb_test, y_imdb_test, verbose=0)
print(f"\nTest accuracy (best transformer): {acc_transformer:.4f}")

---

## Section 9: Best Practices & Common Pitfalls

### Search Space Design

| Rule | Rationale |
|---|---|
| Use **log scale** for learning rates | LRs span orders of magnitude; log scale gives uniform coverage |
| **Narrow the range** after an initial pass | Spend budget where good configs are likely |
| Separate **architecture** from **training** hps | Architecture search requires more trials; training hps can be swept cheaply |
| **Freeze irrelevant dims** with `hp.Fixed` | Reduces true search dimensionality |
| Avoid **interdependent hps** in a single flat space | E.g., `num_heads` must divide `embed_dim` — handle with conditional logic or a custom space |

### Trial Budget Planning

```
RandomSearch:      num_trials ≥ 20        (Bergstra & Bengio: covers space well)
Hyperband:         max_epochs × factor^3  ≈ budget equivalent to ~10 full runs
Bayesian:          5–10 initial points    + 10–20 informed trials
```

### What NOT to do

- **Tune on the test set** — use a held-out validation split only
- **Over-tune on a small dataset** — the "best" hp may just fit the validation noise
- **Ignore early stopping** — always add it; prevents wasted compute on dying trials
- **Report tuned numbers without re-running** — retrain the best hp on train+val before reporting final test accuracy
- **Tune for too many epochs** — start with a fraction of target epochs (1/5 – 1/3), then full-train the winner

In [ ]:
# ── Final retraining on train+val with best hyperparameters ──────────────────
# Best practice: once tuning is complete, retrain on the full train+val set
# using the best HPs before reporting test set performance.

x_final = np.concatenate([x_train, x_val], axis=0)
y_final = np.concatenate([y_train, y_val], axis=0)

# Rebuild best CNN model with best_hp_hb
final_model = hyperband_tuner.hypermodel.build(best_hp_hb)

history_final = final_model.fit(
    x_final, y_final,
    validation_split=0.1,       # small internal val for early stopping
    epochs=25,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping("val_accuracy", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=3),
    ],
    verbose=1,
)

loss_final, acc_final = final_model.evaluate(x_test, y_test, verbose=0)
print(f"\n── Final retrained model ──")
print(f"  Test accuracy : {acc_final:.4f}")
print(f"  Test loss     : {loss_final:.4f}")

In [ ]:
# ── Training curve of the final model ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_final.history["accuracy"],     label="Train", color="#E64A19")
ax1.plot(history_final.history["val_accuracy"], label="Val",   color="#1565C0", linestyle="--")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.set_title("Final Model — Accuracy"); ax1.legend()

ax2.plot(history_final.history["loss"],     label="Train", color="#E64A19")
ax2.plot(history_final.history["val_loss"], label="Val",   color="#1565C0", linestyle="--")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.set_title("Final Model — Loss"); ax2.legend()

plt.tight_layout()
plt.show()

---

## Summary

| What You Learned | Key Takeaway |
|---|---|
| `hp` object API | `Int`, `Float`, `Choice`, `Boolean`, `Fixed` cover all common search space types |
| `RandomSearch` | Strong baseline; use for initial exploration — cheap and parallelisable |
| `Hyperband` | Best default for expensive models — allocates more compute to promising configs |
| `BayesianOptimization` | Best when trials are expensive and search space is low-dimensional |
| Custom `HyperModel.fit()` | Enables tuning of training logic (schedule type, batch size) not just architecture |
| Transformer tuning | `num_heads` must divide `embed_dim` — always enforce with conditional hp logic |
| Results analysis | Correlation-based importance, score distributions, tuner comparison charts |
| Final retraining | Always retrain winner on train + val before reporting test accuracy |

### Next Steps

- **`kt.oracles.CmaEs`** — CMA-ES optimizer for continuous spaces
- **TensorBoard integration** — `keras_tuner.TensorBoardCallback` for live tracking
- **Distributed tuning** — `kt.oracles.grpc` for multi-machine search
- **AutoML / KerasNLP / KerasCV** — built-in tunable preset models
- **NAS (Neural Architecture Search)** — `kt.applications` for transformer NAS

### References

1. Bergstra & Bengio (2012) — *Random Search for Hyper-Parameter Optimization*
2. Li et al. (2017) — *Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization*
3. Snoek et al. (2012) — *Practical Bayesian Optimization of Machine Learning Algorithms*
4. [Keras Tuner Documentation](https://keras.io/keras_tuner/)